# Freeze a plan (prereg)

This notebook demonstrates `prereg`. It:

1. Scaffolds a plan in the 27 OSF question titles.
2. Runs `check` before anything is frozen, and gets *not frozen* rather than *unchanged*.
3. Appends an amendment to the log, and gets a note refused for spanning two lines.
4. Tries to freeze, and gets refused for want of a commit to anchor to.

Every cell runs the **published package**.

## Install

In [ ]:
import piplite

await piplite.install(["prereg==0.2.0"])
print("installed")

## The shell stand-in

`prereg new study` becomes `cli('prereg.cli', 'new', 'study')`.

In [ ]:
import sys


def cli(module, *args):
    old = sys.argv
    sys.argv = [module.split(".")[0], *args]
    try:
        __import__(module, fromlist=["main"]).main()
    except SystemExit:
        pass
    finally:
        sys.argv = old


print("ready")

## Scaffold a plan

In [ ]:
import os
import pathlib

os.makedirs("/tmp/prereg-demo", exist_ok=True)
os.chdir("/tmp/prereg-demo")

cli("prereg.cli", "new", "study")

## The plan, in OSF's 27 headings

A heading that does not apply is answered `N/A` with a reason. It is never deleted — a
deleted heading and an inapplicable one look identical in a file and completely different
to a reader.

In [ ]:
os.chdir("/tmp/prereg-demo/study")
plan = next(pathlib.Path(".").glob("*.md"))
print(plan.name, "\n")
print("\n".join(plan.read_text().splitlines()[:28]))

## Nothing to check against yet

`check` compares the plan against the hash recorded when it was frozen. There is no hash
yet, so it says so rather than reporting the plan as unchanged.

In [ ]:
cli("prereg.cli", "check")

## The log is append-only, and one line per entry

A note carrying a newline would write a second line that reads exactly like an entry
somebody made. The command refuses it.

In [ ]:
cli(
    "prereg.cli",
    "log",
    "switched the primary outcome to 90-day mortality",
    "--access",
    "no results seen",
)
print("--- a note spanning two lines ---")
cli("prereg.cli", "log", "first line\nfrozen at deadbeefcafe", "--access", "no results seen")

---
## Freezing needs a commit

A freeze names the commit the plan was frozen at. WebAssembly has no process model, so there
is no way to ask git anything, and the plan cannot be anchored.

`prereg` on this page is the published 0.2.0, whose `git()` helper lets the error through.
The version in the repository routes every git call through one helper that answers "git
could not tell me" instead — a missing git, a locked index and a directory outside a
repository all have to read the same, or a freeze records a commit-shaped string in place of
a commit.

In [ ]:
try:
    cli("prereg.cli", "freeze")
except OSError as e:
    print(f"freeze could not run: {e}\n")
    print("Locally, git answers and the freeze records the commit and the plan hash.")